# Adversarial-Text-Attacks-and-Detection

Los ataques adversariales en los modelos de procesamiento de lenguaje natural tienen como objetivo exponer vulnerabilidades alterando sutilmente los textos de entrada para engañar las predicciones del modelo. Se han desarrollado varias recetas de ataque, cada una con estrategias únicas que van desde el reemplazo de sinónimos a nivel de palabra hasta perturbaciones a nivel de carácter. La siguiente tabla compara algunas de las recetas de TextAttack más populares, destacando sus características principales, puntos fuertes y casos de uso típicos.


| Nombre del Ataque | Descripción | Fortalezas | Debilidades | Caso de Uso Típico |
|-------------------|-------------|------------|-------------|--------------------|
| PWWSRen2019       | Ataque de prominencia de palabras ponderado por probabilidad. Reemplaza palabras importantes ponderadas por la sensibilidad del modelo. | Eficaz en perturbaciones a nivel de palabra, rápido. | Puede tener dificultades con contextos complejos. | Pruebas adversariales de clasificación de texto. |
| TextFoolerJin2019 | Utiliza incrustaciones de palabras (*word embeddings*) para reemplazar palabras con sinónimos conservando la semántica. | Conserva el significado semántico, intuitivo. | A veces produce oraciones poco naturales. | Ataques robustos basados en sinónimos. |
| DeepWordBug       | Genera perturbaciones a nivel de carácter como errores tipográficos e intercambios para engañar a los modelos. | Eficaz contra modelos sensibles a errores tipográficos. | Menos eficaz en modelos robustos. | Pruebas de robustez ante errores tipográficos. |
| BAEGarg2019       | Genera ejemplos adversariales enmascarando y reemplazando palabras utilizando predicciones de BERT. | Reemplazos conscientes del contexto. | Computacionalmente costoso. | Ataques adversariales conscientes de la semántica. |
| HotFlip           | Utiliza información de gradiente para intercambiar caracteres en ataques adversariales. | Guiado por gradientes, ataques eficaces a nivel de carácter. | Necesita acceso de caja blanca (*white-box*) a los gradientes. | Pruebas adversariales de caja blanca. |
| TextBugger        | Combina perturbaciones a nivel de carácter y de palabra, incluyendo errores tipográficos y reemplazos de sinónimos. | Versátil, ataques de caja negra (*black-box*). | Puede reducir la legibilidad. | Generación adversarial de caja negra. |

---

| Nombre del Ataque | Texto Original | Ejemplo Adversarial | Cambio Clave |
|-------------------|----------------|---------------------|--------------|
| PWWSRen2019       | La película fue absolutamente maravillosa y me encantó cada minuto. | La película fue absolutamente **ordinaria** y me encantó cada minuto. | maravillosa → ordinaria |
| TextFoolerJin2019 | La película fue absolutamente maravillosa y me encantó cada minuto. | La película fue absolutamente **espléndida** y **adoré** cada minuto. | maravillosa → espléndida, encantó → adoré |
| DeepWordBug       | La película fue absolutamente maravillosa y me encantó cada minuto. | L4 **pel1cula** fue **absolutament3 maravillos4** y me encantó cada minuto. | errores tipográficos de caracteres |

# Librerias

In [ ]:
!pip install \
    textattack==0.3.10 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    datasets==3.6.0 \
    evaluate==0.4.3 \
    sentence-transformers==3.3.1 \
    textblob==0.18.0 \
    torch>=2.0.0 \
    --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


# Iniciar sesión en Hugging Face

Este cuaderno utiliza modelos y conjuntos de datos públicos, por lo que no es necesario iniciar sesión. Si alcanzas los límites de peticiones (*rate limits*) durante la descarga, puedes autenticarte con un token.

Opciones:
* Configurar una variable de entorno `HF_TOKEN`.
* (Kaggle) Proporcionar un archivo JSON en `/kaggle/input/autenti/AUTH nn.json` con un campo `API_KEY`.


In [ ]:
import json
import os
from pathlib import Path

from huggingface_hub import login

# Optional: authenticate to increase rate limits when downloading models/datasets.
# For the public assets used here (GLUE SST-2 and DistilBERT SST-2), login is not required.

token = os.getenv("HF_TOKEN")

# Kaggle-specific fallback: allow reading the token from a mounted dataset, if present.
config_path = Path("/kaggle/input/datasets/bryanyamacruz/api-key-1/API_KEY.json")
if token is None and config_path.exists():
    with config_path.open("r", encoding="utf-8") as f:
        config = json.load(f)
    token = config.get("API_KEY")

if token:
    login(token=token)
    print("Successful login to Hugging Face.")
else:
    print("No Hugging Face token found; continuing without login.")


No Hugging Face token found; continuing without login.


In [ ]:
import logging
import warnings

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    logging as hf_logging,
)

from textattack.attack_recipes import PWWSRen2019
from textattack.models.wrappers import HuggingFaceModelWrapper

# Optional (not used in the baseline code below): semantic-similarity-based signals for detection.
from sentence_transformers import SentenceTransformer, util

from textblob import TextBlob

# Print versions to make runs reproducible when sharing results.
import transformers
import datasets
import evaluate
import textattack
import sentence_transformers
import textblob


# Reduce noise in notebook output.
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


2026-03-24 20:54:56.256237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774385696.494910      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774385696.558573      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774385697.090647      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774385697.090723      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774385697.090726      55 computation_placer.cc:177] computation placer alr

device: cuda


# Adversarial Functions

##  Análisis de Métricas:
### ¿Qué significan los resultados de TextAttack?

Cuando ejecutamos un ataque, la consola nos devuelve un resumen crucial para entender la vulnerabilidad de nuestro modelo. Para repasar en el futuro, estos son los valores en los que debemos fijarnos:

* **Original Accuracy (Precisión Original):** Qué tan bien clasifica el modelo los textos limpios, sin ninguna alteración. Un valor alto (ej. 98%) indica que el modelo base es bueno en su tarea principal.
* **Accuracy under attack (Precisión bajo ataque):** El porcentaje de éxito del modelo *después* de que los textos fueron alterados. La caída entre la precisión original y esta métrica demuestra la fragilidad de la red neuronal.
* **Avg num queries (Promedio de consultas):** Cuántas veces el algoritmo de ataque tuvo que "preguntarle" al modelo para encontrar la falla.
  * *Ojo:* Un número bajo (como en `DeepWordBug`) significa que el modelo es muy fácil de quebrar. Un número alto significa que el atacante tuvo que trabajar mucho.
* **Average perturbed word % (Porcentaje de alteración):** Qué tanta porción del texto tuvo que ser modificada. Si un atacante logra quebrar el modelo cambiando solo un 5% de las palabras, el ataque es mucho más peligroso porque es prácticamente indetectable para un supervisor humano.
* **Succeeded / Failed / Skipped:** * *Succeeded:* El ataque engañó al modelo.
  * *Failed:* El modelo resistió el ataque y predijo correctamente.
  * *Skipped:* El modelo ya se equivocaba con el texto limpio, por lo que no tiene sentido atacarlo.

In [ ]:
def preprocess_text(text: str) -> str:
    """Optional normalization step.

    NOTE: `TextBlob(text).correct()` is slow and can change semantics; keep it off unless you are
    explicitly studying spelling-correction as a defense.
    """

    return str(TextBlob(text).correct())


def generate_adversarial_examples(model_wrapper, dataset, num_examples: int = 20, verbose: bool = True):
    """Generate adversarial examples with TextAttack.

    Returns a list of dicts with keys:
    - `text`: perturbed text (or original text if the attack fails)
    - `label`: ground-truth label from the original example

    Guardrails:
    - TextAttack can emit failed/skipped results; we keep the original text so fine-tuning can proceed.
    """

    from textattack import AttackArgs, Attacker

    attack = PWWSRen2019.build(model_wrapper)
    attack_args = AttackArgs(
        num_examples=num_examples,
        shuffle=True,        # randomize which samples are attacked
        disable_stdout=True, # silence TextAttack internal prints (our prints still show)
    )
    attacker = Attacker(attack, dataset, attack_args)

    results = []
    for i, result in enumerate(attacker.attack_dataset()):
        # TextAttack results differ for success/fail/skip; guard against missing fields.
        try:
            original_text = result.original_text()
        except Exception:
            original_text = None

        try:
            perturbed_text = result.perturbed_text()
        except Exception:
            perturbed_text = None

        original_label = getattr(getattr(result, "original_result", None), "ground_truth_output", None)
        if original_label is None:
            raise ValueError("Missing ground-truth label; check that the TextAttack dataset includes labels.")

        predicted_label = getattr(getattr(result, "perturbed_result", None), "output", None)

        # If the attack fails, fall back to the original text to avoid downstream crashes.
        attack_text = perturbed_text or original_text
        if attack_text is None:
            continue

        if verbose:
            print(f"\nExample {i + 1}")
            print("Original text:", original_text)
            print("Perturbed text:", perturbed_text)
            print("Ground-truth label:", original_label)
            print("Predicted label after attack:", predicted_label)
            print("Attack successful:", original_label != predicted_label)

        results.append({
            "text": attack_text,
            "label": int(original_label),
        })

    return results


class CustomDataset(torch.utils.data.Dataset):
    """Minimal dataset wrapper for Hugging Face `Trainer`.

    Each item returns tokenized tensors + an integer label.
    """

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        # Fixed-length padding simplifies batching but can waste compute; tune `max_length` as needed.
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# Datasets - Models

### Datasets:
El siguiente dataset será que el se utilizará.
| Dataset Task | Description                                  | # Train Examples | # Validation Examples | # Test Examples     | Labels                                         |
|--------------|----------------------------------------------|------------------|-----------------------|---------------------|------------------------------------------------|
| SST-2        | Sentiment classification (positive/negative) | 67,349           | 872                   | 1,821               | 0 = negative, 1 = positive                     |
| MRPC         | Paraphrase detection                         | 3,668            | 408                   | 1,725               | 0 = not paraphrase, 1 = paraphrase             |
| QQP          | Quora Question Pairs (paraphrase detection) | 363,849          | 40,431                | 390,965             | 0 = not duplicate, 1 = duplicate               |
| QNLI         | Question-answer entailment                   | 104,743          | 5,463                 | 5,463               | 0 = not entailment, 1 = entailment             |
| MNLI         | Multi-genre Natural Language Inference      | 392,702          | 9,815 (matched)        | 9,796 (mismatched)  | 0 = contradiction, 1 = neutral, 2 = entailment |
| CoLA         | Acceptability of English sentences           | 8,551            | 1,043                 | 1,063               | 0 = unacceptable, 1 = acceptable               |
| RTE          | Recognizing Textual Entailment               | 2,490            | 277                   | 3,000               | 0 = not entailment, 1 = entailment             |
| WNLI         | Winograd Schema Challenge                    | 635              | 71                    | 146                 | 0 or 1 (coreference resolution)                |

### Models:

Modelos que se utilizaran los ejercicios

| Feature                          | `distilbert-base-uncased-finetuned-sst-2-english`       | `cardiffnlp/twitter-roberta-base-sentiment-latest`          |
|----------------------------------|-----------------------------------------------------------|-------------------------------------------------------------|
| **Architecture**                | DistilBERT (lightweight BERT)                            | RoBERTa (Robustly optimized BERT approach)                  |
| **Pretraining Corpus**          | BooksCorpus + English Wikipedia (via BERT)               | 124M English Tweets                                         |
| **Fine-tuned On**              | SST-2 (Stanford Sentiment Treebank)                      | TweetEval sentiment task                                    |
| **Sentiment Labels**            | 0 = Negative, 1 = Positive                               | 0 = Negative, 1 = Neutral, 2 = Positive                     |
| **Domain Focus**                | General (Movie reviews, formal English)                 | Social media (Twitter-specific)                            |
| **Model Size**                  | ~66M parameters                                          | ~125M parameters                                            |
| **Tokenizer**                   | `distilbert-base-uncased` tokenizer                     | `twitter-roberta-base` tokenizer (handles hashtags, emojis)|
| **Performance (General Text)**  | Good general sentiment classification                   | Weaker on non-social media text                            |
| **Performance (Tweets)**        | Moderate, not optimized for tweets                      | Very strong—trained on tweets                              |
| **Use Case Fit**                | Academic, reviews, formal text                          | Twitter, social listening, short informal text             |


# Modelo distilbert-base-uncased-finetuned-sst-2-english

El modelo **distilbert-base-uncased-finetuned-sst-2-english** se utiliza principalmente para el Análisis de Sentimientos (Sentiment Analysis) en textos en inglés. En términos sencillos, su trabajo es leer una oración y clasificarla en una de dos categorías: Positivo o Negativo.

> **https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english**

###  ¿Por qué usamos `distilbert-base-uncased-finetuned-sst-2-english`?

Este modelo clasifica textos en **Positivo** o **Negativo** (Análisis de Sentimientos). Se eligió como modelo base para este experimento por tres razones clave:

1. **Vulnerabilidad Ideal:** Como fue entrenado solo con textos formales y limpios (reseñas de películas del dataset SST-2), no sabe lidiar con el "ruido" de internet. Esto lo hace el sujeto de pruebas perfecto para demostrar cómo los ataques rompen su precisión.
2. **Métrica de Éxito Clara:** Al ser una clasificación binaria, si el ataque logra que el modelo cambie su predicción de "Positivo" a "Negativo", el éxito del engaño es absoluto y muy fácil de medir.
3. **Eficiencia Computacional:** Al ser una versión "destilada" (ligera) de BERT, es mucho más rápido. Permite ejecutar cientos de ataques y realizar el re-entrenamiento para la defensa (*fine-tuning*) en cuestión de minutos sin agotar la memoria.

# Ejempos de Resultados y análisis de los modelos

## Antes de la Defensa


| Métrica                          | Valor   |
|----------------------------------|---------|
| Number of successful attacks     | 36      |
| Number of failed attacks         | 13      |
| Number of skipped attacks        | 1       |
| Original accuracy                | 98.0%   |
| Accuracy under attack            | 26.0%   |
| Attack success rate              | 73.47%  |
| Average perturbed word %         | 30.03%  |
| Average num. words per input     | 9.04    |
| Avg num queries                  | 15.33   |

## Después de la Defensa

| Métrica                          | Valor   |
|----------------------------------|---------|
| Number of successful attacks     | 20      |
| Number of failed attacks         | 30      |
| Number of skipped attacks        | 0       |
| Original accuracy                | 100.0%  |
| Accuracy under attack            | 60.0%   |
| Attack success rate              | 40.0%   |
| Average perturbed word %         | 44.08%  |
| Average num. words per input     | 9.04    |
| Avg num queries                  | 22.88   |

##  Análisis del Ataque: DeepWordBugGao2018 (Errores Tipográficos)

A diferencia de otros ataques que usan sinónimos, este ataque altera los caracteres de las palabras (ej. *'hide'* -> *'hdie'*, *'redundant'* -> *'Qedundant'*). Esto destruye los tokens que el modelo conoce.

A continuación, la comparativa del rendimiento del modelo antes y después de aplicar **Entrenamiento Adversarial** por 5 épocas:

| Métrica |  Antes de la Defensa |  Después de la Defensa |  Interpretación |
| :--- | :---: | :---: | :--- |
| **Original accuracy** | 98.0% | 100.0% | El modelo sigue entendiendo perfectamente el texto limpio. |
| **Accuracy under attack** | 26.0% | **60.0%** | **¡Gran Mejora!** El modelo aprendió a deducir el contexto a pesar de la mala ortografía. |
| **Attack success rate** | 73.47% | 40.0% | La tasa de éxito del atacante se redujo casi a la mitad. |
| **Avg num queries** | 15.33 | 22.88 | Al defenderse mejor, el modelo forzó al atacante a realizar más intentos (consultas) para lograr engañarlo. |
| **Ataques Fallidos (Failed)** | 13 de 50 | **30 de 50** | El modelo logró resistir y clasificar correctamente la mayoría de los ataques. |

**Conclusión del experimento:** El ataque de nivel de caracteres es inicialmente letal para modelos como DistilBERT porque genera vocabulario desconocido. Sin embargo, el entrenamiento adversarial es sumamente efectivo; con solo 5 épocas, el modelo pasó de sucumbir ante la mayoría de los ataques (36 exitosos) a defenderse de la mayoría de ellos (30 fallidos).

In [ ]:
#En la siguiente funcion se hara la utilizacion del modelo "distilbert-base-uncased-finetuned-sst-2-english", se hará una primera corrida del modelo con epoch=5 //
# attack_examples = 50 // Para saber el comportamiento de la red Neuronal.


def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 50

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 15

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...



Example 1
Original text: hide new secretions from the parental units 
Perturbed text: enshroud Modern secretions from the parental units 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 2
Original text: cross swords with the best of them and 
Perturbed text: thwart swords with the best of them and 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 3
Original text: are more deeply thought through than in most ` right-thinking ' films 
Perturbed text: are more deeply mean through than in most ` right-thinking ' films 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 4
Original text: very good viewing alternative 
Perturbed text: very unspoilt viewing alternative 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 5
Original text: equals the original and in some ways even betters it 
Perturbed text: equals the original and in some ways eve

[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 21 / 29 / 0 / 50: 100%|██████████| 50/50 [00:39<00:00,  1.25it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 21     |
| Number of failed attacks:     | 29     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 58.0%  |
| Attack success rate:          | 42.0%  |
| Average perturbed word %:     | 44.78% |
| Average num. words per input: | 9.04   |
| Avg num queries:              | 96.86  |
+-------------------------------+--------+

Example 1
Original text: hide new secretions from the parental units 
Perturbed text: skin fresh secernment from the maternal units 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 2
Original text: cross swords with the best of them and 
Perturbed text: grumpy swords with the undecomposed of them and 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 3
Original

# Actividades

1. Aumenta el número de ejemplos. ¿Cuáles son tus conclusiones sobre el rendimiento del modelo? ¿El modelo mejora o empeora? *Pista: Cantidad de ejemplos*

2. Aumenta el número de épocas de entrenamiento (*training epochs*). ¿Hay alguna mejora en el modelo?

3. Cambia el atacante a DeepWordBugGao2018. ¿Cuáles son las principales diferencias en comparación con PWWSRen2019? ¿Qué atacante es más difícil de corregir?

4. Cambia el modelo a cardiffnlp/twitter-roberta-base-sentiment-latest. ¿Cuál es el rendimiento? Escribe tus conclusiones sobre todo el proceso.

# Actividades

### 1. Aumenta el número de ejemplos. ¿Cuáles son tus conclusiones sobre el rendimiento del modelo? ¿El modelo mejora o empeora? *Pista: Cantidad de ejemplos*

###  Resultados Actividad 1: Aumento a 100 Ejemplos (50 > 100)

Al incrementar la cantidad de ejemplos a `attack_examples = 100`, exponemos al modelo a una mayor variedad de ataques durante su re-entrenamiento (fine-tuning). A continuación, se comparan las métricas antes y después de aplicar esta "vacuna" más robusta:

| Métrica |  Antes de la Defensa |  Después de la Defensa | Análisis |
| :--- | :---: | :---: | :--- |
| **Original accuracy** | 99.0% | 100.0% | El modelo mantiene su capacidad de leer textos limpios. |
| **Accuracy under attack** | 27.0% | **60.0%** | **Mejora drástica:** El modelo duplica su resistencia frente al ataque. |
| **Attack success rate** | 72.73% | **40.0%** | La tasa de éxito del algoritmo atacante cayó significativamente. |
| **Avg num queries** | 71.58 | 86.95 | Al ser más robusto, el modelo fuerza al atacante a realizar más intentos (consultas) por texto para lograr engañarlo. |
| **Ataques Fallidos (Failed)** | 27 de 100 | **60 de 100** | El modelo pasó de ser engañado fácilmente a defenderse con éxito en la mayoría de los casos. |

**Conclusión:** El modelo **sí mejora significativamente**. A mayor cantidad de ejemplos adversariales en el entrenamiento, la red neuronal deja de memorizar palabras específicas y aprende a evaluar el contexto real de la oración, haciéndose mucho más difícil de quebrar.


### 2. Aumenta el número de épocas de entrenamiento (*training epochs*). ¿Hay alguna mejora en el modelo?



###  Resultados Actividad 2: Aumento de Épocas de Entrenamiento (10 Epochs)

**¿Hay alguna mejora en el modelo?**
**Sí, el modelo presenta una mejora adicional.** Al configurar `num_train_epochs = 10`, le permitimos a la red neuronal realizar más "pasadas" sobre los datos. Esto le ayuda a consolidar mejor el aprendizaje y a ajustar sus pesos con mayor precisión para ignorar los ataques.

A continuación, la comparativa del ataque inicial vs. la defensa tras 10 épocas de entrenamiento:

| Métrica |  Antes de la Defensa |  Después (10 Épocas) |  Análisis |
| :--- | :---: | :---: | :--- |
| **Original accuracy** | 99.0% | 100.0% | El modelo mantiene un rendimiento perfecto en textos sin alterar. |
| **Accuracy under attack** | 27.0% | **64.0%** | **Mejora continua:** Sube al 64% (superando el 60% que se obtuvo entrenando solo con 5 épocas). |
| **Attack success rate** | 72.73% | **36.0%** | La tasa de éxito del atacante se reduce aún más, cayendo al 36%. |
| **Avg num queries** | 71.58 | 86.89 | El modelo se vuelve más "terco"; el atacante debe esforzarse más (casi 87 intentos por texto) para encontrar una debilidad. |
| **Ataques Fallidos (Failed)** | 27 de 100 | **64 de 100** | El modelo logró resistir exitosamente 64 de los 100 ataques generados. |

**Conclusión:**
Aumentar las épocas de entrenamiento mejora la robustez del modelo (pasamos de un 60% de resistencia con 5 épocas a un 64% con 10 épocas). Sin embargo, esta mejora nos enseña un concepto clave en Machine Learning: los **rendimientos decrecientes**. Doblar el tiempo de entrenamiento (de 5 a 10) no dobló la resistencia, solo la mejoró un 4%. Esto indica que 10 épocas es un excelente punto de equilibrio, ya que entrenarlo por demasiadas épocas podría llevar a un problema de *overfitting* (sobreajuste).

### 3. Cambia el atacante a DeepWordBugGao2018. ¿Cuáles son las principales diferencias en comparación con PWWSRen2019? ¿Qué atacante es más difícil de corregir?

###  Resultados Actividad 3: Análisis de DeepWordBugGao2018

**1. ¿Cuáles son las principales diferencias en comparación con PWWSRen2019?**
Al analizar los registros del experimento, las diferencias resaltan en dos aspectos clave:
* **Nivel de alteración:** Mientras que PWWSRen2019 cambia palabras completas por sinónimos cuidando el contexto (nivel de palabra), DeepWordBugGao2018 actúa a nivel de caracteres introduciendo **errores tipográficos aleatorios**. En los ejemplos se observa cómo añade, borra o cambia letras (ej. *'sharply'* -> *'shUarply'*, *'heroes'* -> *'heroey'*).
* **Eficiencia del atacante (Consultas):** DeepWordBug es un ataque mucho más rápido. Solo necesitó un promedio de **16.19 consultas** (Avg num queries) al modelo para encontrar una vulnerabilidad y engañarlo, mientras que PWWSRen2019 requería más de 71 consultas en promedio.

**2. ¿Qué atacante es más difícil de corregir/defender?**
Los datos confirman de manera contundente que **DeepWordBugGao2018 es muchísimo más difícil de corregir.**

Al comparar la defensa del modelo (entrenado durante 10 épocas con 100 ejemplos en ambos casos), los resultados son totalmente opuestos:

| Métrica tras 10 Épocas | Defensa vs PWWSRen2019 | Defensa vs DeepWordBug |  Análisis del resultado |
| :--- | :---: | :---: | :--- |
| **Precisión bajo ataque** | Subió al 64.0% | **Apenas subió al 31.0%** (desde un 23%) | El modelo sigue colapsando ante la mala ortografía. |
| **Tasa de éxito del ataque** | Cayó al 36.0% | **Se mantuvo letal en 69.0%** | El atacante sigue logrando su objetivo casi 7 de cada 10 veces. |
| **Ataques Fallidos (Defensa)**| El modelo resistió 64 | **El modelo solo resistió 31** | La "vacuna" adversarial fue insuficiente. |

**Conclusión:**
La gran dificultad radica en el **tokenizador** de DistilBERT. Aprender a defenderse de sinónimos es manejable porque las palabras nuevas siguen existiendo en el diccionario de la IA. Sin embargo, los errores tipográficos generan tokens completamente rotos o desconocidos. Las combinaciones de errores ortográficos son prácticamente infinitas, por lo que re-entrenar al modelo con solo 100 ejemplos alterados no le proporciona la información suficiente para aprender a generalizar y defenderse del ruido a nivel de caracteres.

### 4. Cambia el modelo a cardiffnlp/twitter-roberta-base-sentiment-latest. ¿Cuál es el rendimiento? Escribe tus conclusiones sobre todo el proceso.

###  Resultados Actividad 4: Cambio a RoBERTa (Twitter Sentiment) y Conclusiones Finales

**1. ¿Cuál es el rendimiento del modelo `twitter-roberta-base-sentiment-latest`?**
Al analizar las métricas, observamos un comportamiento inicialmente extraño que luego mejora drásticamente tras el *fine-tuning*:

* **Rendimiento Inicial (Incompatibilidad de Etiquetas):** Antes de la defensa, el modelo tuvo una precisión original de apenas **56.0%** y se saltaron 22 ataques (skipped). ¿Por qué? Porque este modelo de Twitter fue entrenado originalmente para **3 clases** (0=Negativo, 1=Neutral, 2=Positivo), mientras que nuestro dataset (SST-2) es binario. Al evaluar los textos, el modelo predecía muchas veces "1" (Neutral), equivocándose desde el principio. Bajo ataque, su precisión cayó al **2.0%**.
* **Rendimiento tras la Defensa (La Adaptación):** El *fine-tuning* de 5 épocas hizo magia. El modelo no solo aprendió a clasificar correctamente el formato binario de nuestros datos (alcanzando un **100% de precisión original**), sino que su precisión bajo ataque con errores tipográficos subió al **34.0%**. La tasa de éxito del atacante bajó del 96.43% al 66.0%.

---

###  Conclusiones sobre todo el proceso (Adversarial Attacks & Training)

Al finalizar este laboratorio evaluando diferentes modelos (DistilBERT vs. RoBERTa) y diferentes ataques (PWWSRen2019 vs. DeepWordBug), podemos concluir tres grandes lecciones sobre la Inteligencia Artificial en entornos reales:

1. **La Precisión de Laboratorio es una Ilusión:** Un modelo puede tener un 98% o 100% de precisión en un entorno controlado con textos limpios, pero colapsar al 2% o 20% en cuanto un usuario comete errores ortográficos intencionales o accidentales. Los sistemas de IA son inherentemente frágiles ante datos para los que no fueron preparados.

2. **El "Talón de Aquiles" de los LLMs es el Tokenizador:**
   Los ataques a nivel de palabra (sinónimos) son peligrosos, pero los ataques a nivel de caracteres (como `DeepWordBug`) son devastadores. Alterar una sola letra en una palabra destruye su representación matemática (token). Como la IA ya no reconoce la palabra en su diccionario, pierde por completo el hilo semántico de la oración.

3. **El Entrenamiento Adversarial es Obligatorio (MLOps):**
   La única forma de construir sistemas robustos para producción (como filtros antispam, moderadores de contenido o analizadores de reseñas) es aplicar el Entrenamiento Adversarial. Al inyectar deliberadamente "ruido" (mala ortografía, jerga, sinónimos confusos) en el dataset de entrenamiento, forzamos a la red neuronal a dejar de memorizar palabras específicas y a aprender verdaderamente el contexto y la intención del mensaje.

---

### Actividades

El siguiente codigo se utilizó para responder las actividades N°1, y N°2.

In [ ]:
def main():
    # Configuration knobs for the Activities section.
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"

    # How many examples to attack (also equals the number of adversarial examples we attempt to generate).
    attack_examples = 100

    # How many clean examples to include for adversarial training.
    clean_train_examples = 500

    # Validation subset size for quick iteration.
    val_examples = 100

    # Fine-tuning epochs (increase for Activity 2).
    num_train_epochs = 10

    print("Loading model and dataset...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    # GLUE SST-2: binary sentiment classification.
    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack expects a list of (text, label) pairs.
    from textattack.datasets import Dataset

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    print("Generating adversarial examples...")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    print(f"{len(adv_examples)} adversarial examples generated.")

    # Combine clean and adversarial samples for adversarial training.
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    print("Starting fine-tuning...")

    args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()

    # Persist the defended model locally.
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("Fine-tuned model saved to ./defended_model.")

    # Re-wrap the fine-tuned model and rerun the attack to see whether robustness improved.
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    _ = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )


if __name__ == "__main__":
    main()


Loading model and dataset...


[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Generating adversarial examples...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 72 / 27 / 1 / 100: 100%|██████████| 100/100 [00:55<00:00,  1.81it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 72     |
| Number of failed attacks:     | 27     |
| Number of skipped attacks:    | 1      |
| Original accuracy:            | 99.0%  |
| Accuracy under attack:        | 27.0%  |
| Attack success rate:          | 72.73% |
| Average perturbed word %:     | 28.24% |
| Average num. words per input: | 9.01   |
| Avg num queries:              | 71.58  |
+-------------------------------+--------+



Example 1
Original text: goes to absurd lengths 
Perturbed text: ecstasy to laughable lengths 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 2
Original text: contains no wit , only labored gags 
Perturbed text: moderate no wag , only toil jest 
Ground-truth label: 0
Predicted label after attack: 1
Attack successful: True

Example 3
Original text: enriched by an imaginatively mixed cast of antic spirits 
Perturbed text: enrich by an imaginatively mixed vomit of antic spirits 
Ground-truth label: 1
Predicted label after attack: 0
Attack successful: True

Example 4
Original text: swimming is above all about a young woman 's face , and by casting an actress whose face projects that woman 's doubts and yearnings , it succeeds . 
Perturbed text: swimming is above all about a young woman 's face , and by disgorge an actress whose face projects that woman 's doubts and yearnings , it follow . 
Ground-truth label: 1
Predicted label after attack: 0
Atta

[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


Fine-tuned model saved to ./defended_model.
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  weighted-saliency
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapWordNet
  (constraints): 
    (0): RepeatModification
    (1): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 36 / 64 / 0 / 100: 100%|██████████| 100/100 [01:13<00:00,  1.37it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 36     |
| Number of failed attacks:     | 64     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 64.0%  |
| Attack success rate:          | 36.0%  |
| Average perturbed word %:     | 40.46% |
| Average num. words per input: | 9.01   |
| Avg num queries:              | 86.89  |
+-------------------------------+--------+

Example 1
Original text: goes to absurd lengths 
Perturbed text: lead to ridiculous lengths 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 2
Original text: contains no wit , only labored gags 
Perturbed text: contains no wit , only fag gags 
Ground-truth label: 0
Predicted label after attack: 0
Attack successful: False

Example 3
Original text: enriched by an imaginatively mixed cast of antic s

### Actividades

El siguiente codigo se utilizó para responder las actividades N°3.

In [ ]:
"""
Adversarial Training con DeepWordBugGao2018 // Utilizando el modelo DistilBERT fine-tuned
"""
#Librerias

import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_results import SuccessfulAttackResult
from textattack import AttackArgs, Attacker
from textattack.attack_recipes import DeepWordBugGao2018
from textattack.datasets import Dataset
from textblob import TextBlob


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



def preprocess_text(text: str) -> str:
    """Corrección ortográfica opcional (defensa contra DeepWordBug).
    NOTA: Es lenta y puede cambiar semántica.
    Actívala cambiando USE_SPELL_CORRECTION = True abajo.
    """
    return str(TextBlob(text).correct())


# Cambia a True si quieres activar la corrección ortográfica como defensa
USE_SPELL_CORRECTION = False


def generate_adversarial_examples(
    model_wrapper,
    dataset,
    num_examples: int = 20,
    verbose: bool = True,
):
    """Genera ejemplos adversariales con DeepWordBugGao2018.

    Retorna una lista de dicts con:
    - `text`:  texto perturbado (o texto original si el ataque falla)
    - `label`: etiqueta ground-truth del ejemplo original
    """
    attack = DeepWordBugGao2018.build(model_wrapper)
    attack_args = AttackArgs(
        num_examples=num_examples,
        shuffle=True,        # aleatoriza qué muestras se atacan
        disable_stdout=True, # silencia prints internos de TextAttack
    )
    attacker = Attacker(attack, dataset, attack_args)

    results = []
    for i, result in enumerate(attacker.attack_dataset()):
        # TextAttack puede emitir resultados fallidos/saltados; guardamos con try/except
        try:
            original_text = result.original_text()
        except Exception:
            original_text = None

        try:
            perturbed_text = result.perturbed_text()
        except Exception:
            perturbed_text = None

        original_label = getattr(
            getattr(result, "original_result", None), "ground_truth_output", None
        )
        if original_label is None:
            raise ValueError(
                "Missing ground-truth label; verifica que el dataset de TextAttack incluya etiquetas."
            )

        predicted_label = getattr(
            getattr(result, "perturbed_result", None), "output", None
        )

        # Si el ataque falla, usamos el texto original para no perder el ejemplo
        attack_text = perturbed_text or original_text
        if attack_text is None:
            continue

        # Corrección ortográfica opcional (defensa directa contra DeepWordBug)
        if USE_SPELL_CORRECTION and attack_text:
            attack_text = preprocess_text(attack_text)

        if verbose:
            print(f"\n--- Ejemplo {i + 1} ---")
            print(f"  Original  : {original_text}")
            print(f"  Perturbado: {perturbed_text}")
            print(f"  Label GT  : {original_label}")
            print(f"  Label pred: {predicted_label}")
            print(f"  Exitoso   : {original_label != predicted_label}")

        results.append({
            "text": attack_text,
            "label": int(original_label),
        })

    return results


# ---------------------------------------------------------------------------
# Dataset personalizado para Hugging Face Trainer
# ---------------------------------------------------------------------------

class CustomDataset(torch.utils.data.Dataset):
    """Wrapper mínimo para Hugging Face Trainer.
    Cada ítem retorna tensores tokenizados + etiqueta entera.
    """

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    # ------------------------------------------------------------------ #
    # Parámetros configurables
    # ------------------------------------------------------------------ #
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"
    attack_examples = 100       # ejemplos a atacar
    clean_train_examples = 500  # ejemplos limpios para entrenamiento adversarial
    val_examples = 100          # subconjunto de validación
    num_train_epochs = 10       # épocas de fine-tuning

    # ------------------------------------------------------------------ #
    # Carga de modelo y dataset
    # ------------------------------------------------------------------ #
    print("\n=== Cargando modelo y dataset ===")
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    # TextAttack espera una lista de tuplas (texto, etiqueta)
    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        train_raw["label"][:attack_examples],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    # ------------------------------------------------------------------ #
    # Generación de ejemplos adversariales (ANTES del fine-tuning)
    # ------------------------------------------------------------------ #
    print("\n=== Generando ejemplos adversariales con DeepWordBugGao2018 ===")
    adv_examples = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )
    print(f"\n>>> {len(adv_examples)} ejemplos adversariales generados.")

    # ------------------------------------------------------------------ #
    # Preparación de datos para entrenamiento adversarial
    # ------------------------------------------------------------------ #
    clean_data = [
        {"text": x, "label": y}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples
    print(f"\n>>> Dataset combinado: {len(clean_data)} limpios + {len(adv_examples)} adversariales = {len(combined_data)} total")

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": y}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    # ------------------------------------------------------------------ #
    # Fine-tuning (defensa por entrenamiento adversarial)
    # ------------------------------------------------------------------ #
    print("\n=== Iniciando fine-tuning adversarial ===")
    training_args = TrainingArguments(
        output_dir="./defended_model",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )
    trainer.train()

    # Guarda el modelo defendido
    model.save_pretrained("./defended_model")
    tokenizer.save_pretrained("./defended_model")
    print("\n>>> Modelo fine-tuneado guardado en ./defended_model")

    # ------------------------------------------------------------------ #
    # Re-evaluación con el mismo atacante (DESPUÉS del fine-tuning)
    # ------------------------------------------------------------------ #
    print("\n=== Re-evaluando robustez del modelo defendido ===")
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    adv_examples_after = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )

    # Reporte final
    print("\n" + "=" * 60)
    print("REPORTE FINAL")
    print("=" * 60)
    print(f"Atacante           : DeepWordBugGao2018")
    print(f"Spell correction   : {'ON' if USE_SPELL_CORRECTION else 'OFF'}")
    print(f"Ejemplos atacados  : {attack_examples}")
    print(f"Épocas fine-tuning : {num_train_epochs}")
    print(f"Adv. ejemplos antes: {len(adv_examples)}")
    print(f"Adv. ejemplos dopo : {len(adv_examples_after)}")
    print("=" * 60)


if __name__ == "__main__":
    main()

Using device: cuda

=== Cargando modelo y dataset ===


textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.



=== Generando ejemplos adversariales con DeepWordBugGao2018 ===
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 76 / 23 / 1 / 100: 100%|██████████| 100/100 [00:15<00:00,  6.43it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 76     |
| Number of failed attacks:     | 23     |
| Number of skipped attacks:    | 1      |
| Original accuracy:            | 99.0%  |
| Accuracy under attack:        | 23.0%  |
| Attack success rate:          | 76.77% |
| Average perturbed word %:     | 33.06% |
| Average num. words per input: | 9.01   |
| Avg num queries:              | 16.19  |
+-------------------------------+--------+



--- Ejemplo 1 ---
  Original  : goes to absurd lengths 
  Perturbado: goes to absuWd Clengths 
  Label GT  : 0
  Label pred: 1
  Exitoso   : True

--- Ejemplo 2 ---
  Original  : contains no wit , only labored gags 
  Perturbado: Rcontains no zwit , only albored guags 
  Label GT  : 0
  Label pred: 0
  Exitoso   : False

--- Ejemplo 3 ---
  Original  : enriched by an imaginatively mixed cast of antic spirits 
  Perturbado: nriched by an iaginatively mixed cast of antic spirits 
  Label GT  : 1
  Label pred: 0
  Exitoso   : True

--- Ejemplo 4 ---
  Original  : swimming is above all about a young woman 's face , and by casting an actress whose face projects that woman 's doubts and yearnings , it succeeds . 
  Perturbado: swimming is above all about a young woman 's face , and by casting an actress whose face projects that womna 's doubts and yearnings , it succYeeds . 
  Label GT  : 1
  Label pred: 0
  Exitoso   : True

--- Ejemplo 5 ---
  Original  : are more deeply thought through 

textattack: Unknown if model of class <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.



>>> Modelo fine-tuneado guardado en ./defended_model

=== Re-evaluando robustez del modelo defendido ===
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 69 / 31 / 0 / 100: 100%|██████████| 100/100 [00:19<00:00,  5.15it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 69     |
| Number of failed attacks:     | 31     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 31.0%  |
| Attack success rate:          | 69.0%  |
| Average perturbed word %:     | 42.6%  |
| Average num. words per input: | 9.01   |
| Avg num queries:              | 20.15  |
+-------------------------------+--------+

--- Ejemplo 1 ---
  Original  : goes to absurd lengths 
  Perturbado: goes to absuWd Clengths 
  Label GT  : 0
  Label pred: 1
  Exitoso   : True

--- Ejemplo 2 ---
  Original  : contains no wit , only labored gags 
  Perturbado: conains no wt , only laboreId gasg 
  Label GT  : 0
  Label pred: 0
  Exitoso   : False

--- Ejemplo 3 ---
  Original  : enriched by an imaginatively mixed cast of antic spirits 
  Perturbado: enfriched by an 

In [ ]:
"""
Adversarial Training con DeepWordBugGao2018 + RoBERTa

"""

import os
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack import AttackArgs, Attacker
from textattack.attack_recipes import DeepWordBugGao2018
from textattack.datasets import Dataset
from textblob import TextBlob

# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ---------------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------------

def preprocess_text(text: str) -> str:
    """Corrección ortográfica opcional (defensa contra DeepWordBug)."""
    return str(TextBlob(text).correct())


USE_SPELL_CORRECTION = False


def map_label_sst2_to_roberta(label: int) -> int:
    """SST-2: 0=negative, 1=positive → RoBERTa: 0=negative, 2=positive."""
    return 0 if label == 0 else 2


def generate_adversarial_examples(
    model_wrapper,
    dataset,
    num_examples: int = 20,
    verbose: bool = True,
):
    """Genera ejemplos adversariales con DeepWordBugGao2018."""
    attack = DeepWordBugGao2018.build(model_wrapper)
    attack_args = AttackArgs(
        num_examples=num_examples,
        shuffle=True,
        disable_stdout=True,
    )
    attacker = Attacker(attack, dataset, attack_args)

    results = []
    for i, result in enumerate(attacker.attack_dataset()):
        try:
            original_text = result.original_text()
        except Exception:
            original_text = None

        try:
            perturbed_text = result.perturbed_text()
        except Exception:
            perturbed_text = None

        original_label = getattr(
            getattr(result, "original_result", None), "ground_truth_output", None
        )
        if original_label is None:
            raise ValueError("Missing ground-truth label.")

        predicted_label = getattr(
            getattr(result, "perturbed_result", None), "output", None
        )

        attack_text = perturbed_text or original_text
        if attack_text is None:
            continue

        if USE_SPELL_CORRECTION and attack_text:
            attack_text = preprocess_text(attack_text)

        if verbose:
            print(f"\n--- Ejemplo {i + 1} ---")
            print(f"  Original  : {original_text}")
            print(f"  Perturbado: {perturbed_text}")
            print(f"  Label GT  : {original_label}")
            print(f"  Label pred: {predicted_label}")
            print(f"  Exitoso   : {original_label != predicted_label}")

        results.append({
            "text": attack_text,
            "label": int(original_label),
        })

    return results


# ---------------------------------------------------------------------------
# Dataset personalizado
# ---------------------------------------------------------------------------

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    # ------------------------------------------------------------------ #
    # Parámetros
    # ------------------------------------------------------------------ #
    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    attack_examples = 50
    clean_train_examples = 500
    val_examples = 100
    num_train_epochs = 5

    # ------------------------------------------------------------------ #
    # Carga de modelo y dataset
    # ------------------------------------------------------------------ #
    print("\n=== Cargando modelo RoBERTa y dataset ===")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        ignore_mismatched_sizes=True,
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)

    hf_dataset = load_dataset("glue", "sst2")
    train_raw = hf_dataset["train"]
    val_raw = hf_dataset["validation"]

    sample_for_attack = list(zip(
        train_raw["sentence"][:attack_examples],
        [map_label_sst2_to_roberta(l) for l in train_raw["label"][:attack_examples]],
    ))
    textattack_dataset = Dataset(sample_for_attack)

    # ------------------------------------------------------------------ #
    # Generar ejemplos adversariales ANTES del fine-tuning
    # ------------------------------------------------------------------ #
    print("\n=== Generando ejemplos adversariales ANTES del fine-tuning ===")
    adv_examples_before = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )
    print(f"\n>>> {len(adv_examples_before)} ejemplos adversariales generados (antes).")

    # ------------------------------------------------------------------ #
    # Preparar datos combinados
    # ------------------------------------------------------------------ #
    clean_data = [
        {"text": x, "label": map_label_sst2_to_roberta(y)}
        for x, y in zip(
            train_raw["sentence"][:clean_train_examples],
            train_raw["label"][:clean_train_examples],
        )
    ]
    combined_data = clean_data + adv_examples_before
    print(
        f"\n>>> Dataset combinado: {len(clean_data)} limpios + "
        f"{len(adv_examples_before)} adversariales = {len(combined_data)} total"
    )

    train_dataset = CustomDataset(combined_data, tokenizer)
    val_dataset = CustomDataset(
        [
            {"text": x, "label": map_label_sst2_to_roberta(y)}
            for x, y in zip(
                val_raw["sentence"][:val_examples],
                val_raw["label"][:val_examples],
            )
        ],
        tokenizer,
    )

    # ------------------------------------------------------------------ #
    # Fine-tuning
    # FIX: save_strategy="no" evita guardar checkpoints y agotar el disco
    # ------------------------------------------------------------------ #
    print("\n=== Iniciando fine-tuning adversarial con RoBERTa ===")
    training_args = TrainingArguments(
        output_dir="./defended_model_roberta",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        weight_decay=0.01,
        logging_dir="./logs_roberta",
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="no",            # FIX: no guardar checkpoints intermedios
        load_best_model_at_end=False,  # FIX: requiere save_strategy != "no"
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )
    trainer.train()

    # Guardar solo al final (una sola vez)
    os.makedirs("./defended_model_roberta", exist_ok=True)
    model.save_pretrained("./defended_model_roberta")
    tokenizer.save_pretrained("./defended_model_roberta")
    print("\n>>> Modelo guardado en ./defended_model_roberta")

    # ------------------------------------------------------------------ #
    # Re-evaluación DESPUÉS del fine-tuning
    # ------------------------------------------------------------------ #
    print("\n=== Re-evaluando robustez del modelo defendido ===")
    model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
    adv_examples_after = generate_adversarial_examples(
        model_wrapper,
        textattack_dataset,
        num_examples=attack_examples,
        verbose=True,
    )
    print(f"\n>>> {len(adv_examples_after)} ejemplos adversariales generados (después).")

    # ------------------------------------------------------------------ #
    # Reporte final
    # ------------------------------------------------------------------ #
    print("\n" + "=" * 60)
    print("REPORTE FINAL - RoBERTa vs DeepWordBugGao2018")
    print("=" * 60)
    print(f"Modelo             : {model_name}")
    print(f"Atacante           : DeepWordBugGao2018")
    print(f"Spell correction   : {'ON' if USE_SPELL_CORRECTION else 'OFF'}")
    print(f"Épocas fine-tuning : {num_train_epochs}")
    print(f"Ejemplos atacados  : {attack_examples}")
    print(f"Adv. generados ANTES : {len(adv_examples_before)}")
    print(f"Adv. generados DESPUÉS: {len(adv_examples_after)}")
    reduccion = len(adv_examples_before) - len(adv_examples_after)
    pct = reduccion / max(len(adv_examples_before), 1) * 100
    print(f"Reducción de ataques : {reduccion} ({pct:.1f}%)")
    print("=" * 60)


if __name__ == "__main__":
    main()

Using device: cuda

=== Cargando modelo RoBERTa y dataset ===


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...



--- Ejemplo 1 ---
  Original  : hide new secretions from the parental units 
  Perturbado: hide new secretions from the parental units 
  Label GT  : 0
  Label pred: 1
  Exitoso   : True

--- Ejemplo 2 ---
  Original  : cross swords with the best of them and 
  Perturbado: cross swords with the bset of them and 
  Label GT  : 2
  Label pred: 1
  Exitoso   : True

--- Ejemplo 3 ---
  Original  : are more deeply thought through than in most ` right-thinking ' films 
  Perturbado: are more deeply thought through than in most ` right-thinking ' films 
  Label GT  : 2
  Label pred: 1
  Exitoso   : True

--- Ejemplo 4 ---
  Original  : very good viewing alternative 
  Perturbado: very goof viewing alternative 
  Label GT  : 2
  Label pred: 1
  Exitoso   : True

--- Ejemplo 5 ---
  Original  : equals the original and in some ways even betters it 
  Perturbado: equals the original and in some ways even ibetters it 
  Label GT  : 2
  Label pred: 1
  Exitoso   : True

--- Ejemplo 6 ---
  Origi

textattack: Unknown if model of class <class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.



>>> Modelo guardado en ./defended_model_roberta

=== Re-evaluando robustez del modelo defendido ===
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



[Succeeded / Failed / Skipped / Total] 33 / 17 / 0 / 50: 100%|██████████| 50/50 [00:34<00:00,  1.46it/s]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 33     |
| Number of failed attacks:     | 17     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 34.0%  |
| Attack success rate:          | 66.0%  |
| Average perturbed word %:     | 41.35% |
| Average num. words per input: | 9.04   |
| Avg num queries:              | 21.88  |
+-------------------------------+--------+

--- Ejemplo 1 ---
  Original  : hide new secretions from the parental units 
  Perturbado: ide neU osecretions from the piarental nuits 
  Label GT  : 0
  Label pred: 0
  Exitoso   : False

--- Ejemplo 2 ---
  Original  : cross swords with the best of them and 
  Perturbado: cGross sworPs with the est of them and 
  Label GT  : 2
  Label pred: 0
  Exitoso   : True

--- Ejemplo 3 ---
  Original  : are more deeply thought through than in